In [ ]:
from google.analytics.data_v1beta import BetaAnalyticsDataClient

print("Google Analytics connection library ready!")

In [ ]:
from pathlib import Path

downloads = Path.home() / "Downloads"

for file in downloads.glob("*.json"):
    print(file.name)

In [ ]:
from pathlib import Path
from google.analytics.data_v1beta import BetaAnalyticsDataClient

# Replace this with your actual JSON filename
credentials_path = Path(__import__("os").environ["GOOGLE_APPLICATION_CREDENTIALS"])

client = BetaAnalyticsDataClient.from_service_account_file(
    str(credentials_path)
)

print("GA4 authentication successful!")

In [ ]:
import pandas as pd
from google.analytics.data_v1beta.types import (
    RunReportRequest,
    DateRange,
    Dimension,
    Metric
)

property_id = __import__("os").environ["GA4_PROPERTY_ID"]

request = RunReportRequest(
    property=f"properties/{property_id}",
    date_ranges=[
        DateRange(
            start_date="365daysAgo",
            end_date="yesterday"
        )
    ],
    dimensions=[
        Dimension(name="date")
    ],
    metrics=[
        Metric(name="activeUsers"),
        Metric(name="sessions"),
        Metric(name="screenPageViews")
    ],
    limit=10000
)

response = client.run_report(request)

rows = []

for row in response.rows:
    rows.append({
        "date": row.dimension_values[0].value,
        "active_users": int(row.metric_values[0].value),
        "sessions": int(row.metric_values[1].value),
        "page_views": int(row.metric_values[2].value)
    })

df = pd.DataFrame(
    rows,
    columns=["date", "active_users", "sessions", "page_views"]
)

if not df.empty:
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
    df = df.sort_values("date").reset_index(drop=True)

print("Records retrieved:", len(df))
display(df.head(10))

In [ ]:
print("DATASET SUMMARY")
print(df.describe())

print("\nMISSING VALUES")
print(df.isnull().sum())

print("\nDATE RANGE")
print(df["date"].min(), "to", df["date"].max())

print("\nDUPLICATE DATES")
print(df["date"].duplicated().sum())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(df["date"], df["sessions"])

plt.title("HR Innovate - Daily Website Sessions")
plt.xlabel("Date")
plt.ylabel("Sessions")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from google.analytics.data_v1beta.types import (
    RunReportRequest, DateRange, Dimension, Metric
)

request = RunReportRequest(
    property=f"properties/{__import__('os').environ['GA4_PROPERTY_ID']}",
    date_ranges=[
        DateRange(
            start_date="2020-01-01",
            end_date="yesterday"
        )
    ],
    dimensions=[Dimension(name="year")],
    metrics=[Metric(name="sessions")]
)

response = client.run_report(request)

for row in response.rows:
    print(
        "Year:", row.dimension_values[0].value,
        "| Sessions:", row.metric_values[0].value
    )

In [ ]:
request = RunReportRequest(
    property=f"properties/{__import__('os').environ['GA4_PROPERTY_ID']}",
    date_ranges=[
        DateRange(
            start_date="2022-01-01",
            end_date="yesterday"
        )
    ],
    dimensions=[Dimension(name="date")],
    metrics=[
        Metric(name="activeUsers"),
        Metric(name="sessions"),
        Metric(name="screenPageViews")
    ],
    limit=10000
)

response = client.run_report(request)

rows = [
    {
        "date": row.dimension_values[0].value,
        "active_users": int(row.metric_values[0].value),
        "sessions": int(row.metric_values[1].value),
        "page_views": int(row.metric_values[2].value)
    }
    for row in response.rows
]

df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
df = df.sort_values("date").reset_index(drop=True)

print("Total records:", len(df))
print("First date:", df["date"].min())
print("Last date:", df["date"].max())

In [ ]:
# Work with a copy of our original dataset
ml_df = df.copy()

# Ensure chronological order
ml_df = ml_df.sort_values("date").reset_index(drop=True)

# Calendar features
ml_df["day_of_week"] = ml_df["date"].dt.dayofweek
ml_df["month"] = ml_df["date"].dt.month
ml_df["day_of_year"] = ml_df["date"].dt.dayofyear

# Historical traffic features
ml_df["sessions_lag_1"] = ml_df["sessions"].shift(1)
ml_df["sessions_lag_7"] = ml_df["sessions"].shift(7)

# Average sessions over the previous 7 days
ml_df["sessions_rolling_7"] = (
    ml_df["sessions"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

# Remove rows without sufficient history
ml_df = ml_df.dropna().reset_index(drop=True)

print("Records available for ML:", len(ml_df))

display(ml_df.head(10))

In [ ]:
# Check for missing calendar dates
all_dates = pd.date_range(
    start=df["date"].min(),
    end=df["date"].max(),
    freq="D"
)

missing_dates = all_dates.difference(df["date"])

print("Original records:", len(df))
print("Calendar days:", len(all_dates))
print("Missing dates:", len(missing_dates))
print("Duplicate dates:", df["date"].duplicated().sum())
print("ML records:", len(ml_df))

print("\nMissing dates:")
print(missing_dates[:20])

In [ ]:
from sklearn.model_selection import train_test_split

# Exclude the incomplete opening period
clean_df = df[df["date"] >= "2022-05-16"].copy()
clean_df = clean_df.sort_values("date").reset_index(drop=True)

# Verify dates are continuous
assert len(clean_df) == len(
    pd.date_range(clean_df["date"].min(), clean_df["date"].max())
)

# Rebuild features
clean_df["day_of_week"] = clean_df["date"].dt.dayofweek
clean_df["month"] = clean_df["date"].dt.month
clean_df["day_of_year"] = clean_df["date"].dt.dayofyear

clean_df["sessions_lag_1"] = clean_df["sessions"].shift(1)
clean_df["sessions_lag_7"] = clean_df["sessions"].shift(7)

clean_df["sessions_rolling_7"] = (
    clean_df["sessions"].shift(1).rolling(7).mean()
)

ml_df = clean_df.dropna().reset_index(drop=True)

features = [
    "day_of_week",
    "month",
    "day_of_year",
    "sessions_lag_1",
    "sessions_lag_7",
    "sessions_rolling_7"
]

X = ml_df[features]
y = ml_df["sessions"]

split = int(len(ml_df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training ends:", ml_df.iloc[split-1]["date"])
print("Testing begins:", ml_df.iloc[split]["date"])

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# Prepare daily data
daily = df.copy()
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values("date").set_index("date")

# Remove incomplete opening period
daily = daily.loc["2022-05-16":].asfreq("D")

assert daily["sessions"].notna().all(), \
    "Missing dates need investigation."

# Historical features available at forecast time
def make_features(history):
    s = history["sessions"]
    last_date = history.index[-1]
    next_day = last_date + pd.Timedelta(days=1)

    features = {
        "day_of_week": next_day.dayofweek,
        "month": next_day.month,
        "day_of_year": next_day.dayofyear,
        "lag_1": s.iloc[-1],
        "lag_7": s.iloc[-7],
        "lag_14": s.iloc[-14],
        "mean_7": s.iloc[-7:].mean(),
        "mean_28": s.iloc[-28:].mean(),
        "std_28": s.iloc[-28:].std(),
    }

    return features

# Construct training examples
X_rows = []
y_rows = []

for i in range(28, len(daily) - 7 + 1):
    history = daily.iloc[:i]
    future = daily["sessions"].iloc[i:i+7]

    X_rows.append(make_features(history))
    y_rows.append(future.to_numpy())

X = pd.DataFrame(X_rows)
y = np.array(y_rows)

# Train model
model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
)

model.fit(X, y)

# Forecast next seven days
latest = pd.DataFrame([make_features(daily)])
predictions = model.predict(latest)[0]

future_dates = pd.date_range(
    daily.index[-1] + pd.Timedelta(days=1),
    periods=7,
    freq="D"
)

forecast = pd.DataFrame({
    "Date": future_dates,
    "Predicted Sessions": np.maximum(
        0, np.rint(predictions)
    ).astype(int)
})

print("HR INNOVATE: 7-DAY TRAFFIC FORECAST")
display(forecast)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

recent = daily["sessions"].tail(30)

plt.plot(
    recent.index,
    recent.values,
    label="Actual sessions",
    marker="o"
)

plt.plot(
    forecast["Date"],
    forecast["Predicted Sessions"],
    label="7-day forecast",
    marker="o",
    linestyle="--"
)

plt.title("HR Innovate: 7-Day Website Traffic Forecast")
plt.xlabel("Date")
plt.ylabel("Daily Sessions")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()